In [1]:
import re
import math
import random
from collections import Counter, defaultdict

In [2]:
def to_sentences(text):
    return [
        s.strip()
        for s in re.split(r"[.!?]+", text)
        if s.strip()
    ]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

with open("corpus.txt", "r") as file:
    sents = to_sentences(file.read())

print("Number of sentences:", len(sents))
print("First sentence:", sents[0])

Number of sentences: 60
First sentence: I drink coffee every morning before work


In [3]:
def build(sents, n):
    """Map each (n-1)-word context to a Counter of following words."""
    foll = defaultdict(Counter)

    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]

        for i in range(len(toks) - n + 1):
            context = tuple(toks[i:i + n - 1])
            next_word = toks[i + n - 1]
            foll[context][next_word] += 1

    return foll

In [4]:
foll = build(sents, n=2)

print("Words following 'morning':")
print(foll[("morning",)])

print("\nTotal occurrences after 'morning':")
print(sum(foll[("morning",)].values()))

Words following 'morning':
Counter({'</s>': 8, 'train': 2, 'coffee': 2, 'before': 1, 'helps': 1, 'with': 1, 'trains': 1})

Total occurrences after 'morning':
16


In [5]:
def prob(context_word, word):
    c = foll[(context_word,)]
    total = sum(c.values())

    return c[word] / total if total else 0.0

In [6]:
print("P(coffee | morning) =", round(prob("morning", "coffee"), 3))
print("P(train  | morning) =", round(prob("morning", "train"), 3))

P(coffee | morning) = 0.125
P(train  | morning) = 0.125


In [7]:
def suggest_next(context_word, k=3):
    c = foll[(context_word,)]
    total = sum(c.values())

    if total == 0:
        return []

    return [
        (word, round(count / total, 3))
        for word, count in c.most_common(k)
    ]

In [8]:
for w in ["i", "the", "morning", "good", "drink"]:
    print(f"after {w!r:9s}->", suggest_next(w))

after 'i'      -> [('drink', 0.125), ('read', 0.125), ('walk', 0.125)]
after 'the'    -> [('morning', 0.103), ('weekend', 0.088), ('park', 0.074)]
after 'morning'-> [('</s>', 0.5), ('train', 0.125), ('coffee', 0.125)]
after 'good'   -> [('book', 0.6), ('coffee', 0.4)]
after 'drink'  -> [('coffee', 0.5), ('tea', 0.5)]


In [9]:
print("Top 1:", suggest_next("the", k=1))
print("Top 3:", suggest_next("the", k=3))
print("Top 6:", suggest_next("the", k=6))

Top 1: [('morning', 0.103)]
Top 3: [('morning', 0.103), ('weekend', 0.088), ('park', 0.074)]
Top 6: [('morning', 0.103), ('weekend', 0.088), ('park', 0.074), ('city', 0.044), ('train', 0.044), ('rain', 0.044)]


In [10]:
for word in ["morning", "good", "drink", "the"]:
    suggestions = suggest_next(word, k=100)
    total_probability = sum(p for _, p in suggestions)

    print(word, "->", round(total_probability, 3))

morning -> 0.998
good -> 1.0
drink -> 1.0
the -> 1.004


In [11]:
print(suggest_next("spaceship"))
print(prob("spaceship", "coffee"))

[]
0.0


## Observations — Lesson 3

1. MLE converts raw n-gram counts into probabilities by dividing
   each next-word count by the total count of its context.

2. Words that occur more frequently after a context receive higher
   probabilities and appear earlier in autocomplete suggestions.

3. The probabilities of all possible next words for a seen context
   sum to approximately 1.

4. An unseen context receives no suggestions because the model
   has no observations for it.

5. A bigram model only considers the immediately preceding word,
   so it cannot distinguish different meanings of a word based
   on longer context.